In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df = pd.read_parquet("data/batch_total/results.parquet")
flat = df.reset_index()

In [ ]:
# Best F1 and MCC per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best_f1 = group.loc[group["f1_score"].idxmax()]
    best_mcc = group.loc[group["mcc"].idxmax()]
    print(
        f"{bench:25s} "
        f"F1={best_f1['f1_score']:.4f} (k={best_f1['k']})  "
        f"MCC={best_mcc['mcc']:.4f} (k={best_mcc['k']})"
    )

In [ ]:
# Full results table sorted by F1
flat.sort_values("f1_score", ascending=False)[
    [
        "benchmark",
        "k",
        "sae_l0",
        "true_l0",
        "precision",
        "recall",
        "f1_score",
        "mcc",
        "explained_variance",
    ]
].head(12)

## F1 & MCC vs k across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "k"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="k",
    y="score",
    color="benchmark",
    facet_col="metric",
    category_orders={"metric": ["f1_score", "mcc"]},
    markers=True,
    labels={
        "k": "k (BatchTopK)",
        "score": "Score",
        "benchmark": "Benchmark",
    },
    title="F1 & MCC vs k by Distribution (BatchTopK)",
    height=500,
    width=1100,
)
fig.show()

## Precision & Recall vs k across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "k"],
    value_vars=["precision", "recall"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="k",
    y="score",
    color="benchmark",
    facet_col="metric",
    category_orders={"metric": ["precision", "recall"]},
    markers=True,
    labels={
        "k": "k (BatchTopK)",
        "score": "Score",
        "benchmark": "Benchmark",
    },
    title="Precision & Recall vs k by Distribution (BatchTopK)",
    height=500,
    width=1100,
)
fig.show()

## F1 heatmap: benchmark × k

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["F1 Score", "MCC"],
)

for i, metric in enumerate(["f1_score", "mcc"]):
    pivot = flat.pivot_table(values=metric, index="benchmark", columns="k")
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(c) for c in pivot.columns],
            y=pivot.index.tolist(),
            colorscale="Viridis",
            showscale=(i == 1),
            text=pivot.values.round(3),
            texttemplate="%{text}",
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    height=400,
    width=1100,
    title_text="Benchmark × k (BatchTopK)",
)
fig.update_xaxes(title_text="k")
fig.show()

## sae_l0 vs true_l0 across distributions

In [ ]:
fig = px.scatter(
    flat,
    x="true_l0",
    y="sae_l0",
    color="benchmark",
    symbol="k",
    hover_data=["f1_score", "mcc"],
    labels={
        "true_l0": "True L0",
        "sae_l0": "SAE L0",
        "benchmark": "Benchmark",
        "k": "k",
    },
    title="SAE L0 vs True L0 (each point = one config)",
    height=500,
    width=800,
)
# Add y=x reference line
max_val = max(flat["true_l0"].max(), flat["sae_l0"].max())
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val,
    line=dict(dash="dash", color="gray"),
)
fig.show()